# 06 - XLM-R out-of-fold predictions on Colab (Part 9A: CV / OOF mode)

The tri-modal **fusion** (Part 9A) needs, for every train+val page, XLM-R class probabilities from a model that **never saw that page** (out-of-fold, OOF). This notebook trains **5 fold models** on the shared grouped CV folds (`data/splits/v0.3/cv_folds.csv`) and keeps **only their predictions** - no weights to download.

For each fold *k*: train on the other 4 folds **minus an inner-validation slice** (cluster-pure, ~20 %, chosen on the laptop) → early stopping on that inner slice (**never** the test split - the bundle has no test pages at all) → predict fold *k* → delete the weights. Same hyperparameters as Part 6A (`configs/xlmr.yaml`), one seed (Part 6A's final seed, 43).

Made on the laptop:
```
python scripts/train_xlmr.py export-bundle --version v0.3 --cv   # -> data/tmp/xlmr_cv_bundle_v0.3.zip (~0.4 MB)
```

**Steps:** 1 GPU → 2 upload → 3 install → 4 (optional) smoke → 5 train 5 folds → 6 inner-val summary → 7 zip + download (a few KB) → laptop: `import-oof` + `run_ablation.py`.

**Time on a free T4:** Part 6A took 290-350 s per model on 322 pages; a fold model trains on ~250 pages → **~4-5 min per fold, ~20-25 min for 5 folds**, ~30 min in total with installs and the 1.1 GB base-model download. A finished fold is skipped on a re-run, so a disconnect only loses the fold in progress.

## 1. GPU runtime
**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.** The cell must print a GPU name.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

## 2. Upload the CV bundle
Choose `xlmr_cv_bundle_v0.3.zip` (from `data/tmp/` on the laptop). It is unpacked to `/content/cv_bundle`.

In [ ]:
import os, zipfile
from google.colab import files

VERSION = "v0.3"
uploaded = files.upload()  # pick data/tmp/xlmr_cv_bundle_v0.3.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall("/content/cv_bundle")
os.chdir("/content/cv_bundle")
print(open("README.txt").read())

## 3. Install pinned packages
Same pins as Part 6A (the laptop's transformers / tokenizers / accelerate / safetensors / sentencepiece). Colab's CUDA torch is kept.

In [ ]:
!pip -q install -r requirements-colab.txt
import transformers, accelerate, torch
print("transformers", transformers.__version__, "| accelerate", accelerate.__version__, "| torch", torch.__version__)

## 4. (Optional) smoke check - fold 0, 1 training step
Downloads `xlm-roberta-base` once and checks the pipeline. Output goes to `smoke_oof/` (thrown away; `import-oof` refuses smoke runs anyway).

In [ ]:
!python run_train.py --cv --smoke --folds 0 --out smoke_oof

## 5. Train the 5 fold models (predictions only)
Each fold prints its train / inner-val / held-out sizes, the best **inner-validation** macro-F1, the epoch and the time. If the runtime disconnects, re-run steps 2-3 and this cell: finished folds are skipped.

**Out of memory?** In `configs/xlmr.yaml` set `batch_size: 8` and `gradient_accumulation: 2` (same effective batch) and re-run.

In [ ]:
import time
started = time.time()
!python run_train.py --cv --out oof_out/{VERSION}
print(f"total: {(time.time() - started) / 60:.1f} min")

## 6. Inner-validation summary (no test page is involved)

In [ ]:
import json
meta = json.load(open(f"oof_out/{VERSION}/oof_meta.json"))
for m in meta["fold_metrics"]:
    print(f"fold {m['fold']}: train {m['n_train']}, inner val {m['n_inner_val']}, held out {m['n_heldout']} | "
          f"inner-val macro-F1 {m['best_inner_val_macro_f1']:.3f} at epoch {m['best_epoch']} | {m['training_seconds']:.0f} s")
print("complete:", meta["complete"], "| seed", meta["seed"], "|", meta["gpu_name"])

## 7. Zip + download (a few KB: predictions + metadata only)

In [ ]:
import zipfile
from pathlib import Path

root = Path(f"oof_out/{VERSION}")
out = Path(f"/content/xlmr_oof_{VERSION}.zip")
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(root.rglob("*")):
        if path.is_file():
            z.write(path, path.relative_to(root.parent).as_posix())
print(out, f"{out.stat().st_size / 1024:.1f} KB")
files.download(str(out))

## 8. Back on the laptop
```
python scripts/train_xlmr.py import-oof --zip <Downloads>/xlmr_oof_v0.3.zip --version v0.3
python scripts/run_ablation.py --version v0.3 --reuse-oof
```
`import-oof` refuses a zip whose dataset version, split / CV-fold fingerprints, classes or page ids differ from the laptop's, an incomplete run, or a smoke run. It adds the test probabilities of the Part 6A **final** model (re-run on the local CPU) and writes `data/processed/fusion/v0.3/oof_text_xlmr.csv`; `run_ablation.py` then includes Text XLM-R in every row.